# PMM Dynamic Screener — NonKYC Public REST

This notebook screens **NONKYC** markets for **PMM Dynamic (Predictive Market Making)** using only public market-data endpoints. It is a **pre-ingestion gate**: the output is a ranked shortlist plus a candle-ingestor manifest, selected `BASE-QUOTE` pairs, and rule-estimate metadata.

By default this notebook screens **all quote assets** available on NonKYC (USDT, XMR, BTC, USDC, etc.). To restrict to a single quote, set `QUOTE_ASSET` to e.g. `'USDT'`. To screen a specific set, use comma-separated values like `'USDT,XMR'`. The notebook uses the documented public REST endpoints for markets, tickers, order books, candles, and trades.

Stop-ship stance: a pair passing this notebook is **fit for research and ingestion only** until it survives candle-quality checks, walk-forward validation, and live microstructure review.


In [1]:
import os
import sys
import subprocess
import logging
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

CANDIDATE_DIRS = [
    Path("/quants-lab/research_notebooks/market_lab/pmm_dynamic"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PMM_DIR = next((p.resolve() for p in CANDIDATE_DIRS if (p / "pyproject.toml").exists() and (p / "pmm_lab").exists()), None)
if PMM_DIR is None:
    raise FileNotFoundError("Could not locate the pmm_dynamic project root. Open this notebook from inside the pmm_dynamic repo.")

if str(PMM_DIR) not in sys.path:
    sys.path.insert(0, str(PMM_DIR))
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(PMM_DIR), "--quiet"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"PMM project root: {PMM_DIR}")


from pmm_lab.screener import NonKYCPublicScreener, default_nonkyc_config, export_screening_artifacts
from pmm_lab.screener.common import compute_coarse_scores, select_shortlist
from pmm_lab.screener.nonkyc_public import NONKYC_BASE_URL


PMM project root: /quants-lab/research_notebooks/market_lab/pmm_dynamic


## 1. Configuration

Edit the universe, thresholds, and output path here. Keep the defaults conservative unless you explicitly want a wider exploratory net.


In [2]:
# ============================================================
# USER CONFIG — edit these, then Run All
# ============================================================
QUOTE_ASSET = "*"
INTERVAL = "5m"
UNIVERSE_TOP_K = 3000
FINAL_TOP_N = 15
CANDLE_LIMIT = 288
DEPTH_LIMIT = 200
RECENT_TRADE_LIMIT = 500

# Optional pair overrides.
# Keep INCLUDE_SYMBOLS empty to use the full eligible universe.
INCLUDE_SYMBOLS = []
EXCLUDE_SYMBOLS = []

OUTPUT_ROOT = (
    PMM_DIR / "artifacts" / "screener" / "nonkyc" /
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

cfg = default_nonkyc_config()
cfg.quote_asset = QUOTE_ASSET
cfg.interval = INTERVAL
cfg.universe_top_k = UNIVERSE_TOP_K
cfg.final_top_n = FINAL_TOP_N
cfg.candle_limit = CANDLE_LIMIT
cfg.depth_limit = DEPTH_LIMIT
cfg.recent_trade_limit = RECENT_TRADE_LIMIT
cfg.include_symbols = tuple(INCLUDE_SYMBOLS)
cfg.exclude_symbols = tuple(EXCLUDE_SYMBOLS)

# Conservative defaults. Loosen only if you explicitly want more exploratory coverage.
cfg.min_quote_volume_24h = 50000.0
cfg.max_spread_bps = 120.0
cfg.min_top_of_book_quote = 5.0
cfg.min_depth_10bps_quote = 0.0
cfg.max_last_trade_age_sec = 3600.0
cfg.min_recent_trade_count = 40
cfg.min_candle_count = 220
cfg.min_candle_coverage_ratio = 0.9
cfg.max_zero_volume_fraction = 0.3
cfg.min_natr_bps = 12.0
cfg.max_natr_bps = 400.0

print("Screener config")
display(pd.Series(cfg.as_dict()).to_frame("value"))
print(f"Output root: {OUTPUT_ROOT}")


Screener config


,value
connector,nonkyc
quote_asset,*
interval,5m
universe_top_k,3000
final_top_n,15
candle_limit,288
depth_limit,200
recent_trade_limit,500
request_pause_sec,0.2
timeout_seconds,30.0


Output root: /quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635


## 2. Build the cheap universe snapshot

This step uses only batch endpoints to discover the broad market universe and assign a coarse score before any per-pair enrichment calls are made.


In [3]:
screener = NonKYCPublicScreener(cfg)

universe = screener.build_universe()
universe = compute_coarse_scores(universe)
shortlist = select_shortlist(universe, cfg)

summary = pd.DataFrame([
    {
        "connector": cfg.connector,
        "quote_asset": cfg.quote_asset,
        "interval": cfg.interval,
        "universe_rows": len(universe),
        "shortlist_rows": len(shortlist),
    }
])
display(summary)

preview_cols = [
    c for c in [
        "trading_pair",
        "exchange_symbol",
        "quote_volume_24h",
        "spread_bps",
        "coarse_score",
        "price_tick_estimate",
        "amount_step_estimate",
        "status_detail",
    ]
    if c in universe.columns
]

display(universe.loc[:, preview_cols].head(20))
print(f"Shortlist for detailed enrichment: {len(shortlist)}")
display(shortlist.loc[:, preview_cols].head(min(25, len(shortlist))))


,connector,quote_asset,interval,universe_rows,shortlist_rows
0,nonkyc,*,5m,345,345


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,XMR-USDT,XMR/USDT,1.231704e+06,6.247676,79.069921,1.000000e-02,0.001000,active
1,LTC-USDT,LTC/USDT,5.395154e+05,16.560861,77.309306,1.000000e-02,0.000100,active
2,DOGE-USDT,DOGE/USDT,2.000480e+05,13.949246,76.817824,1.000000e-05,0.010000,active
3,NKYC-USDT,NKYC/USDT,1.592436e+05,13.261867,76.253242,1.000000e-06,0.000100,active
4,BTC-USDC,BTC/USDC,8.422135e+05,32.182250,76.033181,1.000000e-02,0.000001,active
5,BNB-USDT,BNB/USDT,1.165307e+06,34.956792,75.547928,1.000000e-02,0.000100,active
6,ETH-USDT,ETH/USDT,4.780444e+06,44.458961,75.162217,1.000000e-02,0.000010,active
7,SEI-USDT,SEI/USDT,9.096892e+04,18.709074,74.711925,1.000000e-04,0.010000,active
8,USDC-USDT,USDC/USDT,1.095369e+06,44.979759,74.292016,1.000000e-04,0.010000,active
9,SHIB-USDT,SHIB/USDT,1.694595e+05,33.277870,74.146580,1.000000e-09,1.000000,active


Shortlist for detailed enrichment: 345


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,XMR-USDT,XMR/USDT,1.231704e+06,6.247676,79.069921,1.000000e-02,0.001000,active
1,LTC-USDT,LTC/USDT,5.395154e+05,16.560861,77.309306,1.000000e-02,0.000100,active
2,DOGE-USDT,DOGE/USDT,2.000480e+05,13.949246,76.817824,1.000000e-05,0.010000,active
3,NKYC-USDT,NKYC/USDT,1.592436e+05,13.261867,76.253242,1.000000e-06,0.000100,active
4,BTC-USDC,BTC/USDC,8.422135e+05,32.182250,76.033181,1.000000e-02,0.000001,active
5,BNB-USDT,BNB/USDT,1.165307e+06,34.956792,75.547928,1.000000e-02,0.000100,active
6,ETH-USDT,ETH/USDT,4.780444e+06,44.458961,75.162217,1.000000e-02,0.000010,active
7,SEI-USDT,SEI/USDT,9.096892e+04,18.709074,74.711925,1.000000e-04,0.010000,active
8,USDC-USDT,USDC/USDT,1.095369e+06,44.979759,74.292016,1.000000e-04,0.010000,active
9,SHIB-USDT,SHIB/USDT,1.694595e+05,33.277870,74.146580,1.000000e-09,1.000000,active


## 3. Run the detailed screener

This step enriches the shortlist with order-book depth, recent trades, and candles, then applies hard gates plus the final PMM-oriented score.


In [4]:
run = screener.screen_from_universe(universe)
final_df = run.final.copy()
selected_df = run.selected.copy()

status_counts = pd.DataFrame([
    {
        "enriched_rows": len(final_df),
        "selected_rows": len(selected_df),
        "pass_rate": float(selected_df.shape[0] / final_df.shape[0]) if len(final_df) else 0.0,
    }
])
display(status_counts)

final_cols = [
    c for c in [
        "trading_pair",
        "screen_score",
        "passed_filters",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "rejection_reason",
    ]
    if c in final_df.columns
]
display(final_df.loc[:, final_cols].head(min(50, len(final_df))))


/opt/conda/envs/quants-lab/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


,enriched_rows,selected_rows,pass_rate
0,345,7,0.02029


,trading_pair,screen_score,passed_filters,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,n_candles,coverage_ratio,zero_volume_fraction,natr_bps_mean,efficiency_ratio,rejection_reason
0,NKYC-USDT,89.810384,True,1.592436e+05,13.261867,97.288628,97.288628,200,8.615142,288,0.996540,0.000000,24.149011,0.033334,
1,TRX-USDT,89.244011,True,4.240028e+05,68.238213,1070.462358,0.000000,200,8.213220,288,0.989691,0.000000,20.809770,0.088571,
2,SOL-USDT,88.788668,True,1.464301e+06,70.284115,98.560800,0.000000,200,1.448521,288,0.996540,0.000000,26.642549,0.038040,
3,BNB-USDT,87.800989,True,1.165307e+06,34.956792,137.764432,0.000000,200,1.524300,288,1.000000,0.006944,20.790782,0.009099,
4,LTC-USDT,87.351500,True,5.395154e+05,16.560861,6.820080,6.820080,200,23.232283,288,0.996540,0.000000,18.663473,0.014512,
5,USDC-USDT,87.010777,True,1.095369e+06,44.979759,75.443148,0.000000,200,10.530616,288,0.979592,0.000000,14.696859,0.002457,
6,AAVE-USDT,82.785319,True,5.631383e+04,64.210365,390.417600,0.000000,200,313.381151,288,0.969697,0.000000,24.629778,0.035280,
7,XMR-USDT,93.519404,False,1.231704e+06,6.247676,1.008690,2.353650,200,7.770344,288,0.993103,0.000000,25.362196,0.070898,top_of_book_quote<5
8,BTC-USDT,88.850475,False,8.205776e+06,74.344495,2.443624,0.000000,200,0.878878,288,0.993103,0.000000,19.099589,0.056616,top_of_book_quote<5
9,BCH2-USDT,86.905446,False,1.108356e+04,11.736141,3.217374,4.573898,200,5.733006,288,0.993103,0.000000,66.993487,0.153997,quote_volume_24h<50000; top_of_book_quote<5


## 4. Diagnostics

Inspect which pairs passed, which pairs failed, and why. Rejection counts are often more useful than raw rankings because they show whether you are liquidity-bound, spread-bound, or data-quality-bound.


In [5]:
if final_df.empty:
    print("No enriched rows returned.")
else:
    passed = final_df[final_df["passed_filters"]].copy()
    rejected = final_df[~final_df["passed_filters"]].copy()

    print(f"Passed: {{len(passed)}} | Rejected: {{len(rejected)}}")

    if not passed.empty:
        display(
            passed.loc[:, [
                c for c in [
                    "trading_pair",
                    "screen_score",
                    "quote_volume_24h",
                    "spread_bps",
                    "top_of_book_quote",
                    "sym_depth_quote_10bps",
                    "recent_trade_count",
                    "last_trade_age_sec",
                    "natr_bps_mean",
                    "efficiency_ratio",
                ] if c in passed.columns
            ]].head(cfg.final_top_n)
        )

    if not rejected.empty:
        reject_counts = (
            rejected["rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        display(reject_counts.head(20))


Passed: {len(passed)} | Rejected: {len(rejected)}


,trading_pair,screen_score,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,natr_bps_mean,efficiency_ratio
0,NKYC-USDT,89.810384,1.592436e+05,13.261867,97.288628,97.288628,200,8.615142,24.149011,0.033334
1,TRX-USDT,89.244011,4.240028e+05,68.238213,1070.462358,0.000000,200,8.213220,20.809770,0.088571
2,SOL-USDT,88.788668,1.464301e+06,70.284115,98.560800,0.000000,200,1.448521,26.642549,0.038040
3,BNB-USDT,87.800989,1.165307e+06,34.956792,137.764432,0.000000,200,1.524300,20.790782,0.009099
4,LTC-USDT,87.351500,5.395154e+05,16.560861,6.820080,6.820080,200,23.232283,18.663473,0.014512
5,USDC-USDT,87.010777,1.095369e+06,44.979759,75.443148,0.000000,200,10.530616,14.696859,0.002457
6,AAVE-USDT,82.785319,5.631383e+04,64.210365,390.417600,0.000000,200,313.381151,24.629778,0.035280


,count
rejection_reason,
top_of_book_quote<5,309
quote_volume_24h<50000,309
coverage_ratio<0.90,200
natr_bps_mean<12,48
spread_bps>120,40
last_trade_age_sec>3600,7
zero_volume_fraction>0.30,3
missing_spread_bps,3
missing_top_of_book_quote,3


## 5. Export artifacts

This writes CSV, JSON, a Markdown report, a candle-ingestor manifest, and an exchange-rules patch for the selected pairs.


In [6]:
artifact_paths = export_screening_artifacts(
    run,
    output_dir=str(OUTPUT_ROOT),
    base_url=NONKYC_BASE_URL,
    trade_limit=cfg.recent_trade_limit,
)

print("Artifacts written")
display(pd.Series(asdict(artifact_paths)).to_frame("path"))


Artifacts written


,path
root_dir,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635
universe_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/universe.csv
shortlist_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/shortlist.csv
final_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/final_screen.csv
selected_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/selected_pairs.csv
selected_pairs_txt,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/selected_pairs.txt
selected_pairs_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/selected_pairs.json
symbol_metadata_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/symbol_metadata.json
candle_ingestor_manifest_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/candle_ingestor_mani...
exchange_rules_patch_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260330_053635/exchange_rules_patch...


## 6. Merge into candle ingestion

The manifest below follows the same `exchanges -> pairs -> intervals` structure already used by your candle-ingest and candle-gap-repair tooling, with slash-formatted exchange pairs and Hummingbot-normalized selected pair files alongside it.


In [7]:
from pathlib import Path

print("Selected Hummingbot pairs")
print("-" * 80)
print(Path(artifact_paths.selected_pairs_txt).read_text())

print("\nCandle ingestor manifest")
print("-" * 80)
print(Path(artifact_paths.candle_ingestor_manifest_yaml).read_text())

print("\nExchange rules patch (estimates only)")
print("-" * 80)
print(Path(artifact_paths.exchange_rules_patch_yaml).read_text())


Selected Hummingbot pairs
--------------------------------------------------------------------------------
NKYC-USDT
TRX-USDT
SOL-USDT
BNB-USDT
LTC-USDT
USDC-USDT
AAVE-USDT


Candle ingestor manifest
--------------------------------------------------------------------------------
backfill_days: 180
request_delay: 0.5
overlap_candles: 2
include_open_candle: false
http:
  timeout_seconds: 30.0
  max_retries: 3
  retry_backoff: 1.8
  user_agent: pmm-lab-screener/0.1
exchanges:
  nonkyc:
    enabled: true
    base_url: https://api.nonkyc.io/api/v2
    pairs:
    - NKYC/USDT
    - TRX/USDT
    - SOL/USDT
    - BNB/USDT
    - LTC/USDT
    - USDC/USDT
    - AAVE/USDT
    intervals:
    - 5m
    trades:
      enabled: true
      limit: 500
      update_recent_candles: false
      recent_window_minutes: 120


Exchange rules patch (estimates only)
--------------------------------------------------------------------------------
connectors:
  nonkyc:
    pairs:
      NKYC-USDT:
        price_tick: